<a href="https://colab.research.google.com/github/sarthak-geek/Neural-Network-SMS-Text-Classifier/blob/main/fcc_rnn_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
# try:
  # %tensorflow_version only exists in Colab.
  # !pip install tf-nightly
# except Exception:
  # pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
# !pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
train_data = pd.read_csv(train_file_path, sep = '\t', header = None, names = ['labels', 'messages'])
test_data = pd.read_csv(test_file_path, sep = '\t', header = None, names = ['labels', 'messages'])
train_data['labels'] = train_data['labels'].replace({'ham':0, 'spam':1})
test_data['labels'] = test_data['labels'].replace({'ham':0, 'spam':1})

In [ ]:
train_labels, train_data = train_data.pop('labels'), train_data['messages']
test_labels, test_data = test_data.pop('labels'), test_data['messages']

In [ ]:
vectorizer = tf.keras.layers.TextVectorization(max_tokens = 8200, output_sequence_length=100)
vectorizer.adapt(train_data)

In [ ]:
len(vectorizer.get_vocabulary())

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(8200, 64),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
hist = model.fit(train_data.to_numpy(), train_labels.to_numpy(), epochs = 7)

In [ ]:
# Evaluate the model on the test data
loss, accuracy = model.evaluate(test_data.to_numpy(), test_labels.to_numpy())

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  ans = model.predict(np.array([pred_text], dtype=object))
  if ans[0][0] <= 0.5:
    return [float(ans[0][0]), 'ham']
  return [float(ans[0][0]), 'spam']


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
